# cuQuantum backend tutorial

This notebook walks through the `"cuquantum"` Qarray backend in jaxquantum.

**When to use it**

- Many-mode systems where the explicit $d^N \times d^N$ Hamiltonian doesn't fit on the GPU.
- Lindblad simulations driven by `mesolve` where each step's matrix-vector product would otherwise materialise huge Kronecker products.

Under the hood, a `Qarray[CuquantumImpl]` carries a `cuquantum.densitymat.jax.OperatorTerm` instead of a dense matrix. Arithmetic (`+`, `*`, `@`, `dag`, `tensor`) builds new `OperatorTerm`s without ever materialising the full tensor product. The solver path replaces the dense matrix multiplications inside the ODE RHS with `cuquantum.densitymat.jax.operator_action`, which runs entirely on GPU.

**Requirements**

- `pip install nvidia-cuquantum-python` (and a CUDA-capable GPU).
- `jax` with x64 enabled.  jaxquantum sets this automatically on import.

## 1. Import and availability check

In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jaxquantum as jqt

# This import will only succeed on a GPU box with cuquantum installed.
from cuquantum.densitymat.jax import OperatorTerm  # noqa: F401

print("cuquantum impl class:", jqt.QarrayImplType.CUQUANTUM.get_impl_class())

/home/shanj/miniconda3/envs/jqt-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuquantum impl class: <class 'jaxquantum.core.cuquantum_impl.CuquantumImpl'>


## 2. Build single-site operators directly

Every `jqt.destroy`, `jqt.create`, `jqt.num`, `jqt.identity`, and the Pauli helpers accept `implementation="cuquantum"`. Each returns a one-mode `Qarray[CuquantumImpl]`.

In [ ]:
d = 4
a = jqt.destroy(d, implementation="cuquantum")
ad = jqt.create(d, implementation="cuquantum")
n = jqt.num(d, implementation="cuquantum")
I = jqt.identity(d, implementation="cuquantum")

print("a.impl_type:", a.impl_type)
print("a.shape:    ", a.shape)
print("a.dims:     ", a.dims)

# to_dense() round-trips the operator to a dense matrix for sanity checks.
# (Don't use this in the hot loop — it defeats the backend's purpose.)
print("\na.to_dense() matches dense backend:",
      jnp.allclose(a.to_dense().data, jqt.destroy(d).data))

## 3. Compose multi-mode operators via `tensor`

`jqt.tensor(...)` dispatches to each impl's `kron`. For cuquantum, the resulting `OperatorTerm` carries the right per-mode structure with mode indices shifted automatically — no $(d^N, d^N)$ matrix is materialised.

In [ ]:
# 3-site hopping Hamiltonian:  J * (a_1^dag a_2 + a_2^dag a_3) + h.c.
J = 1.0

# Per-site operators (one-mode each).
a1 = jqt.destroy(d, implementation="cuquantum")
a2 = jqt.destroy(d, implementation="cuquantum")
a3 = jqt.destroy(d, implementation="cuquantum")
I_  = jqt.identity(d, implementation="cuquantum")

# Embed into the 3-site product space using tensor().
ad1 = jqt.tensor(a1.dag(), I_, I_)
a2_ = jqt.tensor(I_, a2,    I_)
ad2 = jqt.tensor(I_, a2.dag(), I_)
a3_ = jqt.tensor(I_, I_,    a3)

H_hop = J * (ad1 @ a2_ + ad2 @ a3_)
H_hop = H_hop + H_hop.dag()

print("Multi-mode Hamiltonian dims:", H_hop.dims)
print("Multi-mode Hamiltonian shape:", H_hop.shape)
print("impl_type:", H_hop.impl_type)
# `to_dense()` is fine for verification at small sizes — at d=4 with 3 sites,
# the dense matrix is 64x64.
H_hop.to_dense().data.shape

## 4. Set the default backend globally

`Qarray.create` reads `SETTINGS["default_backend"]` when no `implementation=` is passed. Flip it once and every subsequent operator constructor (and any `Qarray.create` call) defaults to cuquantum.

In [ ]:
jqt.SETTINGS["default_backend"] = jqt.QarrayImplType.CUQUANTUM

a_default = jqt.destroy(d)        # no implementation= kwarg
raw       = jqt.Qarray.create(jnp.eye(d, dtype=jnp.complex128))
print("a_default.impl_type:", a_default.impl_type)
print("raw.impl_type:      ", raw.impl_type)

# Restore the default so the rest of the notebook stays explicit.
jqt.SETTINGS["default_backend"] = jqt.QarrayImplType.DENSE

## 5. End-to-end `mesolve` example

A driven, weakly damped 2-qubit system. We build the same Hamiltonian and collapse operators in both backends and confirm the trajectories match.

*Note*: pass `c_ops` for cuquantum as a **Python list** of cuquantum-backed Qarrays. `Qarray.from_list` densifies cuquantum impls (there is no batched `OperatorTerm`).

In [ ]:
d = 2

# Reference (dense)
sx = jqt.sigmax()
I  = jqt.identity(d)
H_dense = 0.5 * (jqt.tensor(sx, I) + jqt.tensor(I, sx))
L1_dense = 0.05**0.5 * jqt.tensor(jqt.sigmam(), I)
L2_dense = 0.05**0.5 * jqt.tensor(I, jqt.sigmam())

# Cuquantum-backed
sxc = jqt.sigmax(implementation="cuquantum")
Ic  = jqt.identity(d, implementation="cuquantum")
smc = jqt.sigmam(implementation="cuquantum")
H_cu = 0.5 * (jqt.tensor(sxc, Ic) + jqt.tensor(Ic, sxc))
L1_cu = 0.05**0.5 * jqt.tensor(smc, Ic)
L2_cu = 0.05**0.5 * jqt.tensor(Ic, smc)

rho0 = jqt.tensor(jqt.basis(d, 0), jqt.basis(d, 0)).to_dm()
tlist = jnp.linspace(0, 5.0, 51)
opts = jqt.SolverOptions.create(progress_meter=False)

ref = jqt.mesolve(
    H_dense, rho0, tlist,
    c_ops=jqt.Qarray.from_list([L1_dense, L2_dense]),
    solver_options=opts,
)
cu = jqt.mesolve(
    H_cu, rho0, tlist,
    c_ops=[L1_cu, L2_cu],   # Python list — see note above
    solver_options=opts,
)
print("reference impl:", ref.impl_type)
print("cuquantum trajectory shape:", cu.data.shape)

In [ ]:
import matplotlib.pyplot as plt

# Population of |00><00| over time, both backends.
p00_ref = jnp.real(ref.data[:, 0, 0])
p00_cu  = jnp.real(cu.data[:, 0, 0])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(tlist, p00_ref, label="dense", lw=2)
ax.plot(tlist, p00_cu, "--", label="cuquantum", lw=2)
ax.set_xlabel("t")
ax.set_ylabel(r"$\langle 00|\rho|00\rangle$")
ax.legend()
fig.tight_layout()
plt.show()

## 6. Parity check

In [ ]:
match = jnp.allclose(ref.data, cu.data, atol=1e-5)
max_err = float(jnp.max(jnp.abs(ref.data - cu.data)))
print(f"Trajectories match (atol=1e-5): {bool(match)}")
print(f"Max abs error:                  {max_err:.2e}")

## Where to go next

- See [`local/lattice_2d_operator_action.py`](../local/lattice_2d_operator_action.py) for a larger 2D qudit lattice that compares dense and cuquantum mesolve at scale.
- The cuquantum backend currently does **not** support batched Qarrays, the propagator's static-H `expm` branch, or sparse-data fallbacks. For those use the `"dense"`, `"sparse_bcoo"`, or `"sparse_dia"` backends.
- Tests in `test/test_cuquantum_backend.py` exercise the construction / arithmetic paths without needing a GPU; the GPU-bound solver tests are tagged with the `cuquantum` pytest marker. CI's default invocation excludes them via `-m "not cuquantum"`.